# UU 来源报价：24 小时连续观察

版本 notebook-observe-1.0；2026-10-13，北京时间。**本次仅交付运行文件，尚未创建观察运行，也未启动采集；由你在本机手动开始。**

本文件是 P03 连续采集验收入口：观察 **AK-47 流金王朝（久经沙场）** 与 **M4A1-S 印花集（崭新出厂）**，通过 SteamDT 公开只读接口每小时留存标明 UU 的商品报价与来源汇总在售/求购报价、数量及更新时间。属于 **L0 来源观察**；汇总数量与交叉报价不能据此解释为可成交深度或套利。

数据自动写入私有 SQLite 与压缩原始响应，不依赖 notebook 输出保存。默认 24 小时、间隔 60 分钟、数据目录存储停止预算 64 MiB（软预算，最后响应或数据库记账可能略超后停止），到首次创建时固定的截止时间自动结束。

先按下面单元逐个运行。首次创建须把配置中的 `START_NEW_RUN` 改为 `True`；创建后把它改回 `False`，恢复时复用原运行。运行期间保持电脑联网、唤醒及 VS Code/kernel 运行；屏幕可以关闭，睡眠、关机或退出 VS Code 会使采集停下。详见 [运行说明](../../docs/usage/OBSERVE_24H.md)。


## 1. 加载项目与 Python 环境

使用 VS Code 的 Python、Jupyter 扩展，右上角选择本机 **Python 3.11 或以上**的 kernel。缺少 kernel 时在该 Python 环境安装 `ipykernel`。本工程采集器不额外依赖 pandas、数据库服务器或 API 密钥。此单元仅检查本地环境。


In [ ]:
from pathlib import Path
from pprint import pprint
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("请选择 Python 3.11 或以上的 notebook kernel。")

project_candidates = [Path.cwd(), *Path.cwd().parents, Path("F:/Codex/CS_YouOweMe")]
PROJECT_ROOT = next(
    (path.resolve() for path in project_candidates
     if (path / "scripts/src/cs_youoweme").is_dir() and (path / "AGENTS.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("未找到项目。请用 VS Code 打开 CS_YouOweMe 文件夹，或修改上面的项目路径。")

source_path = str(PROJECT_ROOT / "scripts/src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
sys.dont_write_bytecode = True

from cs_youoweme.history.observe import create_run, collect, inspect_run, load_last_run

print("项目：", PROJECT_ROOT)
print("Python：", sys.version.split()[0])
print("解释器：", sys.executable)
print("本单元未创建运行，也未请求市场接口。")


## 2. 配置与预检查

第一次开始：将下方 `START_NEW_RUN = False` 改为 `True`。创建完成后把本单元的文字改回 `False`，再次打开或重启 kernel 时就会恢复已有运行。不要同时从 notebook 和终端启动同一个运行。

参数只在创建时生效；恢复时使用原清单参数与原截止时间，修改这里不会延长旧运行。


In [ ]:
HOURS = 24
INTERVAL_MINUTES = 60
MAX_STORAGE_MB = 64
START_NEW_RUN = False  # 首次改为 True；创建后把本行改回 False。

last_run = load_last_run()
pprint({
    "hours": HOURS,
    "interval_minutes": INTERVAL_MINUTES,
    "storage_stop_budget_mib": MAX_STORAGE_MB,
    "start_new_run": START_NEW_RUN,
    "last_run": str(last_run) if last_run else None,
    "specimens": [
        "AK-47 | The Oligarch (Field-Tested)",
        "M4A1-S | Printstream (Factory New)",
    ],
    "scope": "SteamDT UU L0 source observation",
})
print("预检查不访问网络。")


## 3. 创建一次运行，或恢复原运行

首次运行没有历史指针且 `START_NEW_RUN=False` 时，本单元会提示修改配置，不会自动采集。创建时固定起止时刻并保存恢复指针。下一单元才开始网络采集。


In [ ]:
if START_NEW_RUN:
    run_dir = create_run(
        hours=HOURS,
        interval_minutes=INTERVAL_MINUTES,
        max_storage_mb=MAX_STORAGE_MB,
    )
    START_NEW_RUN = False
    print("已创建：", run_dir)
    print("请把配置单元的 START_NEW_RUN 改回 False，再保存 notebook。")
else:
    run_dir = load_last_run()
    if run_dir is None:
        raise RuntimeError("还没有运行。首次开始请在配置单元设置 START_NEW_RUN=True，然后重跑配置和本单元。")
    print("复用已有运行：", run_dir)

pprint(inspect_run(run_dir))


## 4. 连续采集

执行后这个单元会长期显示正在运行；每轮输出进度，轮次之间等待。默认每小时采两件商品的当前来源快照：共 24 个采样时点，从创建时到第 23 小时，随后等待原定第 24 小时截止。恢复时跳过已错过的采样时点，只继续剩余窗口；睡眠或停机期间的历史快照无法补回。本轮不抓历史 K 线，不能还原采样之间的高低价、分钟跳价、挂单与撤单。

**主动停止：点击 notebook 顶部的 Interrupt Kernel（中断内核）**，等待单元停止，再运行报告单元。若 kernel 崩溃或已关闭，重启后依次运行 1、2、3、4，保持 `START_NEW_RUN=False`。如果原截止时间已过，只查看报告。

遇到 HTTP 401、403、429 或认证/风控阻止时会停止访问并记录原因，恢复同一运行不会自动重试受限入口。本采集器不调用已受限的 UU 直接接口。


In [ ]:
if "run_dir" not in globals() or run_dir is None:
    raise RuntimeError("请先运行创建/恢复单元。")

print("开始或恢复：", run_dir, flush=True)
collection_result = collect(run_dir, progress=lambda message: print(message, flush=True))
pprint(collection_result)


## 5. 查看报告（只读）

采集完成、中断或提前停止后均可查看。连续验收看成功轮次、错过时点、来源更新时间、重复/过期快照、异常和磁盘增量；保留 null、零值和求购高于在售的标记。24 小时样本用于检查工程稳定性，不证明有稳定收益或交易机会。

可记下 `run_dir`，把运行结果交给下一轮复核；无需把原始数据复制到 notebook。


In [ ]:
if "run_dir" not in globals() or run_dir is None:
    run_dir = load_last_run()
if run_dir is None:
    raise RuntimeError("还没有可查看的运行，请先完成创建。")

summary = inspect_run(run_dir)
print("运行目录：", run_dir)
pprint(summary, sort_dicts=False)


相关资料：[字段契约](../../docs/data/UU_FIELD_CONTRACT.md)、[P03 前置核验](../../docs/research/P03_VERIFICATION_REPORT.md)、[VS Code notebook 官方说明](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)、[Python kernel 官方说明](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management)。
